# 온라인 쇼핑 세션 데이터 전처리 - KNN / Tree 파이프라인

이 노트북은 `../docs/DataPreProcessing.md`에 기술된 2단 전처리 구조를 그대로 실행한다.

1. **공통 전처리** - 데이터 로드 -> 결측치 확인 -> 중복 확인/제거 -> 타깃 확인 -> train/test split
2. **KNN 파이프라인** - 컬럼 축소 -> PageValues 누수 판단 -> Pipeline 입력 CSV 저장
3. **Tree(DT/RF) 파이프라인** - PageValues 처리 -> 파생 변수 -> One-Hot -> ID 유지

모델 학습은 포함하지 않는다. 최종 산출물은 다음 4쌍이다.

- `X_train_knn`, `X_test_knn`, `y_train_knn`, `y_test_knn`
- `X_train_tree`, `X_test_tree`, `y_train_tree`, `y_test_tree`

KNN의 `log1p`, `StandardScaler`, `SMOTE`는 여기서 CSV로 굳히지 않고 KNN 모델 노트북의 `imblearn.pipeline.Pipeline` 안에서 fold별로 수행한다.

---

**의존성**: `numpy`, `pandas`, `scikit-learn`. KNN 모델 노트북에서만 `imbalanced-learn`을 사용한다.

In [23]:
# 수치/데이터프레임 기본
import numpy as np
import pandas as pd

# 분할은 stratify로 양성 비율 유지
from sklearn.model_selection import train_test_split

# 모든 무작위 단계의 재현성 확보
RANDOM_STATE = 42
pd.set_option('display.max_columns', 40)

## 1. 공통 전처리

세 모델이 같은 train/test 쌍을 쓰도록, split까지는 한 번만 수행한다. 완전 중복 행은 같은 세션이 학습과 평가에 동시에 들어가는 누수를 만들 수 있으므로 split 전에 제거한다. fit이 필요한 변환은 split 이후에 두어 누수를 방지한다.

### 1-1. 데이터 로드

In [24]:
# data/의 원본 CSV 로드. 행 12,330 / 열 18(타깃 포함) 확인.
df = pd.read_csv('../data/online_shoppers_intention.csv')
print(f'데이터 크기: {df.shape}')
df.head()

데이터 크기: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [25]:
# dtype 확인: 수치형/Boolean/문자열(Month, VisitorType) 구분 파악
df.dtypes

Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object

### 1-2. 결측치 확인

UCI 정제본이라 결측 없음을 확인.

In [26]:
# UCI 정제본이라 결측 0개여야 정상. imputation 단계가 불필요함을 검증.
missing = df.isnull().sum()
print(f'전체 결측 셀: {missing.sum()}')
missing[missing > 0] if missing.sum() > 0 else '결측 없음'

전체 결측 셀: 0


'결측 없음'

### 1-3. 중복 행 확인·제거

18개 컬럼이 모두 같은 완전 중복 행은 동일 세션으로 간주한다. split 이후에 제거하면 이미 train/test 양쪽으로 퍼진 중복을 막지 못하므로, 반드시 split 전에 제거한다.

In [27]:
# 완전 중복 125행 제거: 원본 12,330행 중 약 1.0%.
# 기존 split 기준으로는 중복 22개 그룹이 train/test 양쪽에 걸릴 수 있어 평가 누수에 해당.
duplicate_rows = df.duplicated().sum()
duplicate_group_rows = df.duplicated(keep=False).sum()
duplicate_groups = df[df.duplicated(keep=False)].drop_duplicates().shape[0]

print(f'완전 중복 추가 행 수: {duplicate_rows}')
print(f'중복 그룹 포함 전체 행 수: {duplicate_group_rows}')
print(f'중복 그룹 수: {duplicate_groups}')

df = df.drop_duplicates().reset_index(drop=True)
print(f'중복 제거 후 데이터 크기: {df.shape}')

완전 중복 추가 행 수: 125
중복 그룹 포함 전체 행 수: 201
중복 그룹 수: 76
중복 제거 후 데이터 크기: (12205, 18)


### 1-4. 타깃 변수 확인

`Revenue` 양성 비율이 약 15.5%로 불균형하다. stratify와 클래스 가중치/오버샘플링이 필요한 근거.

In [28]:
# 타깃을 int로 캐스팅 (Boolean→0/1). 단조 변환이라 누수 위험 없음 → 공통 단계에 둠.
y_full = df['Revenue'].astype(int)

# 양성 비율 약 15.5%인 불균형 확인. stratify와 SMOTE/class_weight의 근거.
print('Revenue 분포 (count):')
print(y_full.value_counts())
print('\nRevenue 분포 (ratio):')
print(y_full.value_counts(normalize=True).round(4))

Revenue 분포 (count):
Revenue
0    10297
1     1908
Name: count, dtype: int64

Revenue 분포 (ratio):
Revenue
0    0.8437
1    0.1563
Name: proportion, dtype: float64


### 1-5. train/test split (stratify)

모든 모델이 동일한 분할을 공유. 양성 비율을 학습/평가 양쪽에서 유지하기 위해 `stratify=y`.

In [29]:
# 타깃을 분리한 X. 인코딩/스케일은 split 이후에 (테스트 통계 누수 방지).
X_full = df.drop(columns=['Revenue'])

# 80:20 분할. stratify=y로 학습/평가 양쪽에서 양성 비율(~15.5%) 유지.
# 세 모델이 모두 같은 분할을 공유하도록 random_state 고정.
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_full,
)
print(f'X_train: {X_train.shape},  y_train 양성 비율: {y_train.mean():.4f}')
print(f'X_test : {X_test.shape},  y_test  양성 비율: {y_test.mean():.4f}')

X_train: (9764, 17),  y_train 양성 비율: 0.1563
X_test : (2441, 17),  y_test  양성 비율: 0.1565


## 2. KNN 전처리 파이프라인

차원 최소화와 거리 의미 보존이 핵심이다. 단, CV 누수를 막기 위해 fit이 필요한 변환은 이 노트북에서 실행하지 않는다.

1. `Month`, ID 변수 4종, `VisitorType` 제거
2. `VisitorType` -> `is_new_visitor` Boolean 한 비트로 압축
3. `PageValues` 누수 점검 후 제거
4. `Weekend`만 0/1 정수로 정리
5. `log1p` -> `StandardScaler` -> `SMOTE` -> `KNN`은 KNN 모델 노트북의 `imblearn Pipeline`에서 fold별 적용

### 2-1. 컬럼 선택

In [30]:
# 공통 분할 결과를 복사해 KNN 전용으로 가공.
X_tr_knn = X_train.copy()
X_te_knn = X_test.copy()

# VisitorType은 One-Hot으로 +3차원 만들지 않고
# 가장 강한 신호인 "신규 방문자 여부" 한 비트로 압축 (구매율 New 24.9% vs Returning 13.9%).
X_tr_knn['is_new_visitor'] = (X_tr_knn['VisitorType'] == 'New_Visitor').astype(int)
X_te_knn['is_new_visitor'] = (X_te_knn['VisitorType'] == 'New_Visitor').astype(int)

# 제거 대상:
#  - Month: 거리 의미가 약함 (1월-12월 거리=11이 부자연스러움), 신호는 다른 행동 지표에 간접 반영됨
#  - OperatingSystems/Browser/Region/TrafficType: ID 정수에 거리 주면 잡음
#  - VisitorType: is_new_visitor로 환원했으므로 원본은 제거
drop_cols = [
    'Month',
    'OperatingSystems', 'Browser', 'Region', 'TrafficType',
    'VisitorType',
]
X_tr_knn = X_tr_knn.drop(columns=drop_cols)
X_te_knn = X_te_knn.drop(columns=drop_cols)

print('KNN용 컬럼:')
print(X_tr_knn.columns.tolist())

KNN용 컬럼:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Weekend', 'is_new_visitor']


### 2-2. PageValues 누수 점검 + 제거

GA에서 `PageValues`는 거래가 발생한 후 역계산되는 값으로, 예측 시점에 알기 어렵다.
분포 검증으로 의심 정황을 확인한 뒤 제거한다.

In [31]:
# PageValues 누수 점검: 0/>0 그룹의 구매율 격차를 직접 확인
# 격차가 비정상적으로 크면 "거래 이후 역계산된 값"일 가능성이 높음 (예측 시점에 알 수 없는 정보)
check = pd.DataFrame({
    'count':         [(X_tr_knn['PageValues'] == 0).sum(),
                      (X_tr_knn['PageValues'] >  0).sum()],
    'revenue_ratio': [y_train[X_tr_knn['PageValues'] == 0].mean(),
                      y_train[X_tr_knn['PageValues'] >  0].mean()],
}, index=['PageValues == 0', 'PageValues > 0'])
print('누수 의심 점검: PageValues 구간별 구매율')
print(check.round(4))
print('\n→ PageValues > 0 그룹의 구매율이 비정상적으로 높음 (누수 신호). 제거한다.')

# 메인 모델에서는 제거. 비교용으로 살리려면 이 두 줄을 주석 처리.
X_tr_knn = X_tr_knn.drop(columns=['PageValues'])
X_te_knn = X_te_knn.drop(columns=['PageValues'])

누수 의심 점검: PageValues 구간별 구매율
                 count  revenue_ratio
PageValues == 0   7577         0.0393
PageValues > 0    2187         0.5615

→ PageValues > 0 그룹의 구매율이 비정상적으로 높음 (누수 신호). 제거한다.


### 2-3. Pipeline 입력용 타입 정리

여기서는 `Weekend` Boolean만 0/1로 바꾼다. `log1p`, `StandardScaler`, `SMOTE`는 `GridSearchCV` 내부 fold마다 다시 fit되어야 하므로 CSV 저장 전에 적용하지 않는다.

In [32]:
# Boolean -> int (CSV 저장 및 sklearn 호환). fit이 필요한 변환은 아님.
X_tr_knn['Weekend'] = X_tr_knn['Weekend'].astype(int)
X_te_knn['Weekend'] = X_te_knn['Weekend'].astype(int)

# KNN 모델 노트북에서 Pipeline으로 적용할 후보 컬럼 목록.
knn_log_cols = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'SpecialDay',
]

print(f'KNN Pipeline 입력 shape: train {X_tr_knn.shape}, test {X_te_knn.shape}')
print('KNN 모델 노트북에서 fold별 적용: log1p -> StandardScaler -> SMOTE -> KNN')
X_tr_knn.describe().round(3)

KNN Pipeline 입력 shape: train (9764, 11), test (2441, 11)
KNN 모델 노트북에서 fold별 적용: log1p -> StandardScaler -> SMOTE -> KNN


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,SpecialDay,Weekend,is_new_visitor
count,9764.000,9764.000,9764.000,9764.000,9764.000,9764.000,9764.000,9764.000,9764.000,9764.000,9764.000
mean,2.343,82.220,0.510,34.951,32.102,1207.265,0.020,0.041,0.062,0.234,0.138
std,3.349,179.502,1.283,143.828,44.817,1926.389,0.045,0.046,0.200,0.423,0.345
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.000,0.000,0.000,0.000,8.000,193.467,0.000,0.014,0.000,0.000,0.000
50%,1.000,9.000,0.000,0.000,18.000,617.000,0.003,0.025,0.000,0.000,0.000
75%,4.000,94.725,0.000,0.000,39.000,1483.650,0.017,0.049,0.000,0.000,0.000
max,27.000,3398.750,24.000,2549.375,705.000,63973.522,0.200,0.200,1.000,1.000,1.000


### 2-4. KNN 최종 산출물

`X_train_knn`은 SMOTE/Scaler 적용 전 데이터다. KNN 노트북에서 이 CSV를 읽고 `imblearn.pipeline.Pipeline`을 `GridSearchCV`에 전달한다.

In [33]:
# 최종 KNN 입력. SMOTE/Scaler 적용 전 split 데이터 그대로 저장한다.
X_train_knn = X_tr_knn.copy()
y_train_knn = y_train.copy()
X_test_knn  = X_te_knn.copy()
y_test_knn  = y_test.copy()

print('KNN 최종 산출물 (Pipeline 입력용, SMOTE/Scaler 전)')
print(f'  X_train_knn : {X_train_knn.shape}')
print(f'  X_test_knn  : {X_test_knn.shape}')
print(f'  y_train_knn : {y_train_knn.shape}, 양성 비율 {y_train_knn.mean():.4f}')
print(f'  y_test_knn  : {y_test_knn.shape}, 양성 비율 {y_test_knn.mean():.4f}')
print(f'\n입력 컬럼 ({len(X_train_knn.columns)}개):')
print(list(X_train_knn.columns))

KNN 최종 산출물 (Pipeline 입력용, SMOTE/Scaler 전)
  X_train_knn : (9764, 11)
  X_test_knn  : (2441, 11)
  y_train_knn : (9764,), 양성 비율 0.1563
  y_test_knn  : (2441,), 양성 비율 0.1565

입력 컬럼 (11개):
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'SpecialDay', 'Weekend', 'is_new_visitor']


## 3. Tree (DT / RF) 전처리 파이프라인

정보 최대 보존 + 명시적 비율 + 사후 가지치기 친화 전략.

1. `PageValues` 제거 (KNN과 동일한 누수 결정)
2. 파생 변수 4개 추가
3. `Month`, `VisitorType` One-Hot
4. ID 변수 유지 + `Weekend` int
5. 불균형은 모델 단계에서 `class_weight='balanced'`로 처리 (여기서는 데이터만 준비)

### 3-1. PageValues 제거 (KNN과 일관)

In [34]:
# PageValues는 KNN과 같은 누수 판단 결과로 제거
# 같은 데이터 정의 위에서 비교해야 모델 간 성능 차이가 공정하게 해석됨
X_tr_tree = X_train.drop(columns=['PageValues']).copy()
X_te_tree = X_test.drop(columns=['PageValues']).copy()
print(f'PageValues 제거 후 shape: train {X_tr_tree.shape}, test {X_te_tree.shape}')

PageValues 제거 후 shape: train (9764, 16), test (2441, 16)


### 3-2. 파생 변수 생성

트리가 직접 표현하기 어려운 비율/합 신호를 명시적으로 제공해 분할 깊이를 절약한다.

In [35]:
# 트리는 변수 간 비율/곱을 직접 표현 못 함 → 명시적으로 만들어 분할 깊이 절약
def add_derived(X):
    X = X.copy()
    # 세션 전체 탐색 강도 (페이지 총합)
    X['total_pages'] = (
        X['Administrative'] + X['Informational'] + X['ProductRelated']
    )
    # 세션 전체 체류 시간 합
    X['total_duration'] = (
        X['Administrative_Duration']
        + X['Informational_Duration']
        + X['ProductRelated_Duration']
    )
    # 상품 한 페이지당 평균 체류 (분모에 +1로 0 페이지 방어)
    X['avg_time_per_product'] = (
        X['ProductRelated_Duration'] / (X['ProductRelated'] + 1)
    )
    # 즉시 이탈 vs 점진 이탈 균형 (분모에 epsilon으로 0 방어)
    X['bounce_exit_ratio'] = (
        X['BounceRates'] / (X['ExitRates'] + 1e-6)
    )
    return X

X_tr_tree = add_derived(X_tr_tree)
X_te_tree = add_derived(X_te_tree)

print('파생 변수 추가 완료')
X_tr_tree[['total_pages', 'total_duration',
           'avg_time_per_product', 'bounce_exit_ratio']].describe().round(3)

파생 변수 추가 완료


,total_pages,total_duration,avg_time_per_product,bounce_exit_ratio
count,9764.000,9764.000,9764.000,9764.000
mean,34.955,1324.435,34.912,0.284
std,46.862,2054.214,35.158,0.350
min,0.000,0.000,0.000,0.000
25%,9.000,231.950,16.350,0.000
50%,20.000,701.000,27.442,0.160
75%,42.000,1655.673,43.200,0.500
max,746.000,69921.647,705.500,4.746


### 3-3. One-Hot 인코딩 (`Month`, `VisitorType`)

`drop_first=False`로 모든 범주를 살린다. 트리는 다중공선성에 강하므로 손해 없다.
테스트셋은 학습셋 컬럼 기준으로 정렬해야 한다 (test에만 등장하는 카테고리 방어).

In [36]:
# Month / VisitorType One-Hot
# drop_first=False: 모든 범주를 살림. 트리는 다중공선성에 강하므로 손해 없음
cat_cols = ['Month', 'VisitorType']
X_tr_tree = pd.get_dummies(X_tr_tree, columns=cat_cols, drop_first=False)
X_te_tree = pd.get_dummies(X_te_tree, columns=cat_cols, drop_first=False)

# 테스트셋에만 있거나 빠진 범주가 있을 수 있으므로 train 컬럼 기준으로 정렬
X_te_tree = X_te_tree.reindex(columns=X_tr_tree.columns, fill_value=0)

# get_dummies는 bool로 반환. sklearn 호환을 위해 0/1 정수로 통일
for d in (X_tr_tree, X_te_tree):
    bool_cols = d.select_dtypes(include='bool').columns
    d[bool_cols] = d[bool_cols].astype(int)

print(f'One-Hot 후 shape: train {X_tr_tree.shape}, test {X_te_tree.shape}')

One-Hot 후 shape: train (9764, 31), test (2441, 31)


### 3-4. Boolean 변환 (Weekend)

ID 변수(`OperatingSystems`, `Browser`, `Region`, `TrafficType`)는 그대로 유지한다.
트리가 정보 이득으로 사용 여부를 자동 판단한다.

In [37]:
# Boolean → int (sklearn 호환)
X_tr_tree['Weekend'] = X_tr_tree['Weekend'].astype(int)
X_te_tree['Weekend'] = X_te_tree['Weekend'].astype(int)

# ID 변수는 유지 — 트리는 정보 이득으로 사용 여부를 자동 판단 (KNN과 반대)
print('유지된 ID 변수:')
for c in ['OperatingSystems', 'Browser', 'Region', 'TrafficType']:
    if c in X_tr_tree.columns:
        print(f'  - {c}: 고유값 {X_tr_tree[c].nunique()}개')

유지된 ID 변수:
  - OperatingSystems: 고유값 8개
  - Browser: 고유값 13개
  - Region: 고유값 9개
  - TrafficType: 고유값 19개


### 3-5. Tree 최종 산출물

In [38]:
# 최종 Tree 입력. y는 SMOTE 없이 원본 분포 유지 (class_weight='balanced'로 모델 단계에서 처리).
X_train_tree = X_tr_tree
X_test_tree  = X_te_tree
y_train_tree = y_train
y_test_tree  = y_test

print('Tree 최종 산출물')
print(f'  X_train_tree : {X_train_tree.shape}')
print(f'  X_test_tree  : {X_test_tree.shape}')
print(f'  y_train_tree : {y_train_tree.shape}, 양성 비율 {y_train_tree.mean():.4f}')
print(f'  y_test_tree  : {y_test_tree.shape}, 양성 비율 {y_test_tree.mean():.4f}')
print(f'\n입력 컬럼 ({len(X_train_tree.columns)}개):')
print(list(X_train_tree.columns))

Tree 최종 산출물
  X_train_tree : (9764, 31)
  X_test_tree  : (2441, 31)
  y_train_tree : (9764,), 양성 비율 0.1563
  y_test_tree  : (2441,), 양성 비율 0.1565

입력 컬럼 (31개):
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend', 'total_pages', 'total_duration', 'avg_time_per_product', 'bounce_exit_ratio', 'Month_Aug', 'Month_Dec', 'Month_Feb', 'Month_Jul', 'Month_June', 'Month_Mar', 'Month_May', 'Month_Nov', 'Month_Oct', 'Month_Sep', 'VisitorType_New_Visitor', 'VisitorType_Other', 'VisitorType_Returning_Visitor']


## 4. 산출물 저장

다른 노트북에서 바로 이어서 모델 학습을 할 수 있도록 모델별 CSV 디렉토리에 저장한다. KNN CSV는 `log1p`, `StandardScaler`, `SMOTE` 적용 전 Pipeline 입력이다.

In [39]:
from pathlib import Path

# 노트북을 notebooks/ 안에서 실행하면 부모 폴더가 프로젝트 루트,
# 루트에서 실행하면 현재 폴더가 프로젝트 루트가 된다.
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
knn_csv_dir = project_root / 'models' / 'knn' / 'csv'
tree_csv_dir = project_root / 'models' / 'decision_tree' / 'csv'
rf_csv_dir = project_root / 'models' / 'random_forest' / 'csv'
knn_csv_dir.mkdir(parents=True, exist_ok=True)
tree_csv_dir.mkdir(parents=True, exist_ok=True)
rf_csv_dir.mkdir(parents=True, exist_ok=True)

# KNN: GridSearchCV Pipeline 입력용. SMOTE/Scaler 전 데이터를 저장한다.
# X_train_knn.to_csv(knn_csv_dir / 'X_train_knn.csv', index=False)
# X_test_knn .to_csv(knn_csv_dir / 'X_test_knn.csv',  index=False)
# y_train_knn.to_csv(knn_csv_dir / 'y_train_knn.csv', index=False)
# y_test_knn .to_csv(knn_csv_dir / 'y_test_knn.csv',  index=False)

# Tree 계열: 기존 모델 노트북 입력용.
# X_train_tree.to_csv(tree_csv_dir / 'X_train.csv', index=False)
# X_test_tree .to_csv(tree_csv_dir / 'X_test.csv',  index=False)
# y_train_tree.to_csv(tree_csv_dir / 'y_train.csv', index=False)
# y_test_tree .to_csv(tree_csv_dir / 'y_test.csv',  index=False)

# Random Forest도 같은 Tree 계열 전처리 입력을 사용한다.
# X_train_tree.to_csv(rf_csv_dir / 'X_train.csv', index=False)
# X_test_tree .to_csv(rf_csv_dir / 'X_test.csv',  index=False)
# y_train_tree.to_csv(rf_csv_dir / 'y_train.csv', index=False)
# y_test_tree .to_csv(rf_csv_dir / 'y_test.csv',  index=False)

print(f'KNN CSV files saved : {knn_csv_dir}')
print(f'Tree CSV files saved: {tree_csv_dir}')
print(f'RF CSV files saved  : {rf_csv_dir}')

KNN CSV files saved : e:\shopper-prediction\models\knn\csv
Tree CSV files saved: e:\shopper-prediction\models\decision_tree\csv
RF CSV files saved  : e:\shopper-prediction\models\random_forest\csv


## 5. 요약

| 산출물 | 입력 컬럼 수 | 양성 비율 | 비고 |
|---|---:|---:|---|
| `X_train_knn` | 11 | ~15.6% | SMOTE/Scaler 전, GridSearchCV Pipeline 입력 |
| `X_test_knn`  | 11 | ~15.6% | 동일 |
| `X_train_tree` | ~29 | ~15.6% | One-Hot + 파생변수 4개 |
| `X_test_tree`  | ~29 | ~15.6% | 동일 |

KNN은 `PageValues` 제거와 컬럼 정리까지만 공통 전처리에서 수행한다. `log1p`, `StandardScaler`, `SMOTE`는 KNN 모델 노트북의 `imblearn.pipeline.Pipeline` 안에서 fold별로 적용해야 CV 누수를 막을 수 있다.